# HydraY NNUE - hidden layer 16 → 32

Runtime → **GPU (T4)**, then run the cells in order. Eight stages, as for deep16.

### The question
The champion is `1024 → 16 → 1` (+7.89 over the single-layer net). Here the
hidden layer is **32** wide.

**Only `L1_SIZE` changes.** Same v7 data, same 160 superbatches, same slice order,
same WDL, LR schedule, init, pairwise, 4 king buckets and 8 output buckets. If
this net wins, the width paid for itself; if it loses, the width did not.

### The cost is already paid and measured
The C++ forward is written and checked bit for bit against `sanity_deep.rs` on
29,703 real positions (two random nets, VNNI and non-VNNI paths). With the
champion widened to 32 outputs whose extra weights are zero (same evals, same
tree): forward **74 -> 94 ns per eval (+27%)**, engine NPS **-3.4% (identical tree, slower in 5/5 paired bench6 rounds)**. The net starts in that debt.

### The check that matters, and it comes EARLY
After **stage 1** (superbatch 20) look at the `running loss` and compare it with
the deep16 run at the same point. Unlike deep vs single layer, these two losses
ARE comparable: a 32-wide layer can represent everything the 16-wide one can.
It should be at or below deep16's. Clearly above means something is broken
(wrong clone, wrong trainer), so stop after half an hour instead of four.

Lower loss is necessary, not sufficient: the 8-bucket net had a lower loss and
lost the SPRT.


In [ ]:
# --- helper: qualunque comando fallito ferma il notebook, e l'output si vede ---
import subprocess, os, sys, json

def sh(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FALLITO (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')
print('\nGPU presente. Se la riga sopra non mostra una T4, cambia runtime.')

In [ ]:
# --- Drive + configurazione ---
from google.colab import drive
drive.mount('/content/drive')

import glob
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert hits, f'{name} non trovato su Drive'
    return hits[0]

PARTS = {i: find(f'hydray_v7_part{i}.bin.zst') for i in (1, 2, 3, 4)}
# Taglia attesa DOPO la decompressione, per parte. Una decompressione
# interrotta a meta' produce un file piu' corto e nessun errore: senza questo
# controllo si addestrerebbe in silenzio su dati troncati.
RAW_SIZE = {1: 23_742_906_368, 2: 23_742_906_368,
            3: 23_742_906_368, 4: 23_745_983_264}
for i, p in PARTS.items():
    print(f'parte {i}: {os.path.getsize(p)/2**30:6.2f} GiB compressa  {p}')

NET_ID   = 'hydray-deep32-160sb'
TOTAL_SB = 160          # stesso budget e stessi dati della rete adottata:
                        # l'unica variabile e' l'architettura
STAGE    = 20           # otto tappe
ORDER    = [1, 2, 3, 4, 1, 2, 3, 4]   # ogni fetta girata due volte
TRAINER  = '/content/th/nnue/trainer'
assert len(ORDER) * STAGE == TOTAL_SB and STAGE % 10 == 0

In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')

In [ ]:
# --- clone + architecture check ---
# Not ceremonial: an earlier run trained a different architecture for an hour
# because the clone was wrong, and a checkpoint name says nothing about content.
BRANCH = 'nnue-l2-32'
sh('rm -rf /content/th')
sh(f'git clone --depth 1 --branch {BRANCH} https://github.com/ThomasGhione/HydraY /content/th')

tr = open(f'{TRAINER}/src/bin/trainer_deep.rs').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr, 'trainer_deep.rs is not 1024'
assert 'const L1_SIZE: usize = 32;' in tr, 'trainer_deep.rs hidden layer is not 32'
assert 'const OUTPUT_BUCKETS: usize = 8;' in tr, 'output buckets must stay 8'
sd = open(f'{TRAINER}/src/bin/sanity_deep.rs').read()
assert 'const HIDDEN: usize = 1024;' in sd, 'sanity_deep.rs is not 1024'
assert 'const L1_SIZE: usize = 32;' in sd, 'sanity_deep.rs hidden layer is not 32'
assert 'const INPUT_BUCKETS: usize = 4;' in sd, 'king buckets must stay 4'
print(f'branch {BRANCH}, 1024 -> 32 -> 1, 4 king buckets, 8 output buckets: ok')

sh('apt-get -qq install -y zstd >/dev/null')
st = os.statvfs('/content'); free_gb = st.f_bavail * st.f_frsize / 2**30
print(f'free {free_gb:.1f} GiB, expected peak ~36 GiB (one slice + Drive cache)')
assert free_gb > 45, 'not enough disk'


In [ ]:
# --- helper delle tappe (ESEGUIRE SEMPRE, anche in ripartenza) ---

def load_slice(n):
    """Scompatta la fetta n in /content/data.bin, sostituendo la precedente."""
    if os.path.exists('/content/data.bin'):
        os.remove('/content/data.bin')          # spazio prima, non dopo
    sh(f'zstd -d -T0 --long=27 -c "{PARTS[n]}" > /content/data.bin')
    got = os.path.getsize('/content/data.bin')
    assert got == RAW_SIZE[n], f'fetta {n} troncata: {got} != {RAW_SIZE[n]}'
    print(f'fetta {n}: {got//32/1e6:.1f}M posizioni, taglia verificata', flush=True)

def stage_cmd(end, start, resume_from):
    """⚠️ STAGE_END DEVE STARE ATTACCATO A `cargo`, non in testa alla riga.
    `STAGE_END=40 cd dir && cargo ...` assegna la variabile SOLO a `cd`: cargo
    la riceve vuota, il trainer ignora le tappe e tira dritto fino a TOTAL_SB
    senza salvare niente. E' costato un run intero. Da qui l'`env` esplicito."""
    args = f'/content/data.bin {TOTAL_SB} {NET_ID}'
    if resume_from is not None:
        args += f' {start} checkpoints/{NET_ID}-{resume_from}'
    return (f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH '
            f'env CUDA_PATH=/usr/local/cuda STAGE_END={end} '
            f'cargo run -r --bin trainer_deep --features cuda -- {args}')

def save_to_drive(end):
    """Copia il checkpoint su Drive e VERIFICA che ci sia arrivato davvero: il
    mount di Drive scrive attraverso una cache, quindi un upload mai completato
    passerebbe per riuscito."""
    ck  = f'{TRAINER}/checkpoints/{NET_ID}-{end}'
    dst = f'/content/drive/MyDrive/{NET_ID}-{end}'
    assert os.path.isdir(ck), f'checkpoint mancante in locale: {ck}'
    sh(f'rm -rf {dst} && cp -r {ck} /content/drive/MyDrive/')
    size = lambda p: sum(os.path.getsize(os.path.join(d, f))
                         for d, _, fs in os.walk(p) for f in fs)
    assert os.path.isdir(dst), f'la copia su Drive non esiste: {dst}'
    assert size(dst) == size(ck), f'copia su Drive incompleta: {size(dst)} != {size(ck)}'
    print(f'tappa fino al superbatch {end} su Drive ({size(dst)/2**20:.0f} MiB, verificata)', flush=True)

def run_stages(done=0):
    """Esegue le tappe da `done` in poi. done=0 parte da zero."""
    assert done % STAGE == 0, f'{done} non e un confine di tappa'
    prev = done if done else None
    for k in range(done // STAGE, len(ORDER)):
        end, start, sl = (k+1)*STAGE, k*STAGE + 1, ORDER[k]
        print(f'\n===== tappa {k+1}/{len(ORDER)}: superbatch {start}-{end}, fetta {sl} =====', flush=True)
        load_slice(sl)
        sh(stage_cmd(end, start, prev))
        save_to_drive(end)
        prev = end

In [ ]:
# --- training: otto tappe, fette 1-2-3-4-1-2-3-4 ---
# NON eseguire questa cella in una ripartenza: usa invece la cella in fondo.
run_stages(done=0)

In [ ]:
# --- final check and save to Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
# 6,557,728 B of payload, padded to 64. deep16 is 6,425,664: if that number
# comes out, the wrong branch was trained.
assert 6557728 <= sz < 6557728 + 64, f'size {sz}: NOT the 1024->32->1 net'
print('quantised.bin:', sz, 'bytes - 32-wide hidden layer confirmed\n')

sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity_deep -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print('\n' + '='*70)
print('REFERENCE - the champion (deep16, v7, 160 SB), measured locally:')
print('  startpos            64      middlegame ~24 pieces  897')
print('  KQvK               958      KRPvKR                 126')
print('  extra knight       721      active kings (endgame) 109')
print('  extra queen       1087')
print()
print('Sanity evals catch disasters (broken mirror, absurd values), they do')
print('not predict Elo. The verdict is the SPRT.')
print('='*70)


In [ ]:
# --- RIPARTENZA (usare SOLO se la sessione e' morta a meta') ---
# Come si usa:
#   1. esegui le celle da "helper" fino a "helper delle tappe" compresa;
#   2. NON eseguire la cella del training;
#   3. metti RESUME = True e DONE = ultimo superbatch salvato su Drive.
# La fetta giusta viene ricavata da ORDER: non devi ricordarti dov'era.
#
# Con RESUME = False questa cella non fa niente, cosi' "Esegui tutte" e' sicuro
# (altrimenti, a run finito, ripartirebbe da DONE rifacendo ore di training).

RESUME = False
DONE   = 20      # ultimo superbatch salvato su Drive

if not RESUME:
    print('ripartenza disattivata (RESUME = False) - nessuna azione')
else:
    ck = f'/content/drive/MyDrive/{NET_ID}-{DONE}'
    assert os.path.isdir(ck), f'checkpoint non trovato su Drive: {ck}'
    os.makedirs(f'{TRAINER}/checkpoints', exist_ok=True)
    sh(f'cp -r {ck} {TRAINER}/checkpoints/')
    assert os.path.isdir(f'{TRAINER}/checkpoints/{NET_ID}-{DONE}')
    print(f'ripartenza dal superbatch {DONE} (prossima fetta: {ORDER[DONE//STAGE]})\n')
    run_stages(done=DONE)

## After training

1. **Reorder** the net with `nnue/tools/reorder.cpp` (FENs from real games), as
   the champion was: the sparse l1 path depends on it, and it is exact (the tool
   refuses to write a net whose evals change).
2. Put it in `nnue/net/hydray.nnue` **on `nnue-l2-32`** and `make prod`.
3. **The baseline needs its own binary.** A 32-wide build cannot load a 16-wide
   net (file size check), so `EvalFile` cannot carry the candidate: freeze a
   `dev` build as the baseline and SPRT the two binaries at 4+0.04, threads=1.

## How to read the result

The candidate plays with the NPS cost included, so the number is net of it.

- **Wins**: width pays. Next is 64, or a second hidden layer.
- **Draw**: the gross gain equals the NPS cost. Recovering speed (a better group
  layout in the sparse loop) could tip it; measure at fixed nodes to see the
  gross.
- **Loses**: the layer is not the bottleneck at this budget.
